In [1]:
# 04_ggi.ipynb

# %% [markdown]
# # Cross-Domain AI Governance Gap Analysis
# ## 04. Governance Gap Index (GGI) — refined
#
# Hybrid threshold-distance (option B) + normalized per-component gaps (option A).
#
# For component k under regulation r, with required threshold tau[r,k] and a model score
# s_k oriented so that "higher is better":
#     g[r,k] = max(0, tau[r,k] - s_k) / tau[r,k]          # normalized shortfall in [0,1]
#     GGI[r] = sqrt( sum_k g[r,k]^2 )                     # lower is better (0 = full compliance)
#
# Four component scores (refined from the 2-axis draft):
#   s_PERF : discriminative adequacy    = max(0, (AUC - 0.5) / 0.5)
#   s_DI   : fairness (worst-case DI)   = min(1, min over sensitive attrs of DisparateImpact)
#   s_SP   : fairness (worst-case SPD)  = 1 - max over sensitive attrs of |StatParityDiff|
#   s_EXPL : explainability             = 1 - normalized entropy of |SHAP| distribution
#
# Rationale for refinements:
#  - s_PERF prevents a near-random model from scoring "fully compliant" (GGI 0).
#  - worst-case fairness avoids missing violations that a single representative attribute hides.
#  - entropy-based explainability removes the feature-count confound of top-k concentration.
#
# Thresholds tau encode a modeled regulatory strictness gradient (KR < EU < NAIC),
# not codified legal constants (see paper Methodology).

# %%
# Imports and configuration
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

RANDOM_STATE = 42
REPRESENTATIVE_PROXY = "ps_ind_04_cat"
SHAP_SAMPLE = 1000
os.makedirs(TAB_DIR, exist_ok=True)

# %%
# Regulatory strictness gradient (modeled scenario, not codified constants).
# Required thresholds per component score; all scores oriented so higher = better.
REGULATIONS = {
    "KR_AI_Basic_Act": {"s_PERF": 0.30, "s_DI": 0.80, "s_SP": 0.90, "s_EXPL": 0.50},
    "EU_AI_Act":       {"s_PERF": 0.40, "s_DI": 0.85, "s_SP": 0.92, "s_EXPL": 0.60},
    "NAIC":            {"s_PERF": 0.40, "s_DI": 0.90, "s_SP": 0.95, "s_EXPL": 0.55},
}

# (display_name, key)
DATASETS = [
    ("German Credit",        "german"),
    ("Credit Card Approval", "credit_card_approval"),
    ("Medical Cost",         "insurance"),
    ("Porto Seguro",         "porto_seguro"),
]

# %%
# Load metadata and helpers
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

def load_split(key):
    train = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet"))
    test = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet"))
    return train, test

def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[target_col].astype(int)
    return X, y

# %%
# Fairness rates for a single binary grouping (positive prediction = predicted high-risk)
def group_di_spd(y_pred, priv_mask, unpriv_mask):
    ppr_p = y_pred[priv_mask].mean() if priv_mask.sum() else np.nan
    ppr_u = y_pred[unpriv_mask].mean() if unpriv_mask.sum() else np.nan
    if not (np.isfinite(ppr_p) and np.isfinite(ppr_u)) or ppr_p == 0:
        return np.nan, np.nan
    di = ppr_u / ppr_p
    spd = ppr_u - ppr_p
    return di, spd

# Worst-case fairness across all available sensitive attributes for a dataset.
# For each sensitive attribute, the privileged group is its most frequent value.
def worst_case_fairness(test_df, sensitive_cols, y_pred):
    dis, spds = [], []
    for col in sensitive_cols:
        s = test_df[col]
        if s.nunique(dropna=True) < 2:
            continue
        privileged = s.mode(dropna=True).iloc[0]
        priv_mask = (s == privileged).values
        unpriv_mask = (s != privileged).values & s.notna().values
        di, spd = group_di_spd(y_pred, priv_mask, unpriv_mask)
        if np.isfinite(di):
            dis.append(di)
        if np.isfinite(spd):
            spds.append(spd)
    # Worst DI = furthest below 1 (min); worst SPD = largest absolute gap
    worst_di = min(dis) if dis else np.nan
    worst_abs_spd = max(abs(x) for x in spds) if spds else np.nan
    s_DI = min(1.0, worst_di) if np.isfinite(worst_di) else np.nan
    s_SP = 1.0 - worst_abs_spd if np.isfinite(worst_abs_spd) else np.nan
    return s_DI, s_SP

# %%
# Explainability score: 1 - normalized entropy of the mean |SHAP| distribution.
# Normalized entropy divides by log(n_features), removing the feature-count confound.
def shap_explainability(model, X, sample=SHAP_SAMPLE, seed=RANDOM_STATE):
    if len(X) > sample:
        X = X.sample(sample, random_state=seed)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)
    if isinstance(sv, list):
        sv = sv[-1]
    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    if total == 0 or len(mean_abs) <= 1:
        return 0.0
    p = mean_abs / total
    p = p[p > 0]
    entropy = -np.sum(p * np.log(p))
    norm_entropy = entropy / np.log(len(mean_abs))  # in [0,1]
    return float(1.0 - norm_entropy)                # concentration in [0,1]

# %%
# Prevalence-based threshold (matches notebook 03 convention)
def prevalence_threshold(proba, y_train):
    prevalence = float(y_train.mean())
    return np.quantile(proba, 1.0 - prevalence)

# %%
# Train unified baseline per dataset; collect the four oriented component scores.
component_scores = {}

for display_name, key in DATASETS:
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]

    n_pos = int(y_train.sum()); n_neg = int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0

    model = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
    )
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    thr = prevalence_threshold(proba, y_train)
    y_pred = (proba >= thr).astype(int)

    # s_PERF: discriminative adequacy from AUC
    auc = roc_auc_score(y_test.values, proba)
    s_PERF = max(0.0, (auc - 0.5) / 0.5)

    # s_DI, s_SP: worst-case fairness across all sensitive attributes
    s_DI, s_SP = worst_case_fairness(test, META[key]["sensitive_cols"], y_pred)

    # s_EXPL: entropy-based explainability
    s_EXPL = shap_explainability(model, X_test)

    component_scores[display_name] = {
        "domain": META[key].get("domain", ""),
        "AUC": auc,
        "s_PERF": s_PERF, "s_DI": s_DI, "s_SP": s_SP, "s_EXPL": s_EXPL,
    }
    print(f"[{display_name:>22}] AUC={auc:.3f}  s_PERF={s_PERF:.3f}  "
          f"s_DI={s_DI:.3f}  s_SP={s_SP:.3f}  s_EXPL={s_EXPL:.3f}")

# %%
# Component score table
score_df = pd.DataFrame([
    {"domain": v["domain"], "dataset": k, "AUC": v["AUC"],
     "s_PERF": v["s_PERF"], "s_DI": v["s_DI"], "s_SP": v["s_SP"], "s_EXPL": v["s_EXPL"]}
    for k, v in component_scores.items()
]).round(4)
print(score_df.to_string(index=False))
score_df.to_csv(os.path.join(TAB_DIR, "04_component_scores.csv"), index=False)

# %%
# Compute normalized component gaps and GGI for every (dataset x regulation) pair
def component_gap(score, tau):
    if not np.isfinite(score):
        return np.nan
    return max(0.0, tau - score) / tau

ggi_rows = []
for display_name, scores in component_scores.items():
    for reg_name, tau in REGULATIONS.items():
        g_PERF = component_gap(scores["s_PERF"], tau["s_PERF"])
        g_DI = component_gap(scores["s_DI"], tau["s_DI"])
        g_SP = component_gap(scores["s_SP"], tau["s_SP"])
        g_EXPL = component_gap(scores["s_EXPL"], tau["s_EXPL"])

        gaps = [g for g in [g_PERF, g_DI, g_SP, g_EXPL] if np.isfinite(g)]
        ggi = float(np.sqrt(np.sum(np.square(gaps)))) if gaps else np.nan

        ggi_rows.append({
            "domain": scores["domain"], "dataset": display_name, "regulation": reg_name,
            "g_PERF": g_PERF, "g_DI": g_DI, "g_SP": g_SP, "g_EXPL": g_EXPL, "GGI": ggi,
        })

ggi_df = pd.DataFrame(ggi_rows).round(4)
print(ggi_df.to_string(index=False))
ggi_df.to_csv(os.path.join(TAB_DIR, "04_ggi_full.csv"), index=False)

# %%
# Core comparison matrix: GGI by dataset x regulation
ggi_matrix = ggi_df.pivot(index="dataset", columns="regulation", values="GGI")
ggi_matrix = ggi_matrix[["KR_AI_Basic_Act", "EU_AI_Act", "NAIC"]].round(4)
print(ggi_matrix.to_string())
ggi_matrix.to_csv(os.path.join(TAB_DIR, "04_ggi_matrix.csv"))

# %%
# Domain-level summary: mean GGI per domain x regulation
domain_summary = (
    ggi_df.groupby(["domain", "regulation"])["GGI"]
    .mean().round(4).reset_index()
    .pivot(index="domain", columns="regulation", values="GGI")
)
domain_summary = domain_summary[["KR_AI_Basic_Act", "EU_AI_Act", "NAIC"]]
print(domain_summary.to_string())
domain_summary.to_csv(os.path.join(TAB_DIR, "04_ggi_by_domain.csv"))

[         German Credit] AUC=0.802  s_PERF=0.603  s_DI=1.000  s_SP=0.907  s_EXPL=0.159
[  Credit Card Approval] AUC=0.690  s_PERF=0.381  s_DI=1.000  s_SP=0.965  s_EXPL=0.234
[          Medical Cost] AUC=0.552  s_PERF=0.105  s_DI=0.661  s_SP=0.899  s_EXPL=0.150
[          Porto Seguro] AUC=0.556  s_PERF=0.112  s_DI=0.000  s_SP=0.963  s_EXPL=0.072
   domain              dataset    AUC  s_PERF   s_DI   s_SP  s_EXPL
   credit        German Credit 0.8016  0.6032 1.0000 0.9072  0.1591
   credit Credit Card Approval 0.6904  0.3807 1.0000 0.9650  0.2337
insurance         Medical Cost 0.5523  0.1046 0.6609 0.8994  0.1502
insurance         Porto Seguro 0.5560  0.1120 0.0000 0.9634  0.0724
   domain              dataset      regulation  g_PERF   g_DI   g_SP  g_EXPL    GGI
   credit        German Credit KR_AI_Basic_Act  0.0000 0.0000 0.0000  0.6818 0.6818
   credit        German Credit       EU_AI_Act  0.0000 0.0000 0.0139  0.7348 0.7350
   credit        German Credit            NAIC  0.0000 0.000